In [0]:
%pip install --upgrade notebook nbformat nbconvert jupyter

# **01 EDA Y PREPROCESAMIENTO DE LA BASE DE DATOS - CYCLA CON PypSpark**

**_-- IMPORTANDO LIBRERIAS NECESARIAS_**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

**_-- LEYENDO CSV's_**

In [0]:
base_path = ""  # reemplazar con el workspace correspondiente

brands_tbl = f"{base_path}brands.csv"  # df_brands
categories_tbl = f"{base_path}categories.csv"
customers_tbl = f"{base_path}customers.csv"
order_items_tbl = f"{base_path}order_items.csv"
orders_tbl = f"{base_path}orders.csv"
products_tbl = f"{base_path}products.csv"
staffs_tbl = f"{base_path}staffs.csv"
stocks_tbl = f"{base_path}stocks.csv"
stores_tbl = f"{base_path}stores.csv"

df_brands_tbl = spark.read.csv(brands_tbl, header=True, inferSchema=True)
df_categories_tbl = spark.read.csv(categories_tbl, header=True, inferSchema=True)
df_customers_tbl = spark.read.csv(customers_tbl, header=True, inferSchema=True)
df_order_items_tbl = spark.read.csv(order_items_tbl, header=True, inferSchema=True)
df_orders_tbl = spark.read.csv(orders_tbl, header=True, inferSchema=True)
df_products_tbl = spark.read.csv(products_tbl, header=True, inferSchema=True)
df_staffs_tbl = spark.read.csv(staffs_tbl, header=True, inferSchema=True)
df_stocks_tbl = spark.read.csv(stocks_tbl, header=True, inferSchema=True)
df_stores_tbl = spark.read.csv(stores_tbl, header=True, inferSchema=True)

**_-- NORMALIZACIÓN DE DATOS_**

In [0]:
display(df_customers_tbl) # NORMALIZAR NULOS CAMPO PHONE

display(df_staffs_tbl) # NORMALIZAR NULO -> GG
 
display(df_stocks_tbl) # SI NO HAY STOCK SIRVE PARA PREDICCIÓN Y ALERTAS

-- _**NULOS**_

In [0]:
# Normalización de NULLS
df_customers_tbl = df_customers_tbl.withColumn(
    "phone",
    F.when(
        (F.col("phone").isNull()) | (F.col("phone") == "null"),
        "No agendado"
    ).otherwise(F.col("phone")).cast("string")
)
display(df_customers_tbl)


df_staffs_tbl = df_staffs_tbl.withColumn(
    "manager_id",
    F.when(
        (F.col("manager_id").isNull()) | (F.col("manager_id") == "NULL"),
        "Administrador"
    ).otherwise(F.col("manager_id")).cast("string")
)
display(df_staffs_tbl)

# **02 GENERACIÓN DE TABLAS TRANSACCIONALES (OLTP)**

Limpieza de Datos

In [0]:
# ==============================
# 1. LIMPIEZA GENERAL
# ==============================

tablas = {
    "brands": df_brands_tbl,
    "categories": df_categories_tbl,
    "customers": df_customers_tbl,
    "order_items": df_order_items_tbl,
    "orders": df_orders_tbl,
    "products": df_products_tbl,
    "staffs": df_staffs_tbl,
    "stocks": df_stocks_tbl,
    "stores": df_stores_tbl
}

# Eliminar duplicados en todas las tablas
for nombre, df in tablas.items():
    tablas[nombre] = df.dropDuplicates()

df_brands_tbl = tablas["brands"]
df_categories_tbl = tablas["categories"]
df_customers_tbl = tablas["customers"]
df_order_items_tbl = tablas["order_items"]
df_orders_tbl = tablas["orders"]
df_products_tbl = tablas["products"]
df_staffs_tbl = tablas["staffs"]
df_stocks_tbl = tablas["stocks"]
df_stores_tbl = tablas["stores"]

In [0]:
# ==============================
# 2. REVISIÓN DE NULOS
# ==============================

def contar_nulos(df, nombre_tabla):
    print(f"Tabla: {nombre_tabla}")
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()

for nombre, df in tablas.items():
    contar_nulos(df, nombre)

In [0]:
# ==============================
# 3. NORMALIZACIÓN DE NULOS
# ==============================

df_customers_tbl = df_customers_tbl.withColumn(
    "phone",
    F.when(
        (F.col("phone").isNull()) | 
        (F.lower(F.col("phone")) == "null") |
        (F.trim(F.col("phone")) == ""),
        "No registrado"
    ).otherwise(F.col("phone"))
)

df_staffs_tbl = df_staffs_tbl.withColumn(
    "manager_id",
    F.when(
        (F.col("manager_id").isNull()) |
        (F.lower(F.col("manager_id").cast("string")) == "null") |
        (F.trim(F.col("manager_id").cast("string")) == ""),
        "Administrador"
    ).otherwise(F.col("manager_id").cast("string"))
)

In [0]:
# ==============================
# 4. LIMPIAR ESPACIOS EN COLUMNAS TEXTO
# ==============================

def limpiar_texto(df):
    for c, tipo in df.dtypes:
        if tipo == "string":
            df = df.withColumn(c, F.trim(F.col(c)))
    return df

df_brands_tbl = limpiar_texto(df_brands_tbl)
df_categories_tbl = limpiar_texto(df_categories_tbl)
df_customers_tbl = limpiar_texto(df_customers_tbl)
df_staffs_tbl = limpiar_texto(df_staffs_tbl)
df_stores_tbl = limpiar_texto(df_stores_tbl)
df_products_tbl = limpiar_texto(df_products_tbl)

In [0]:
display(df_customers_tbl)
display(df_staffs_tbl)
display(df_stocks_tbl)

# **03 TRANSFORMACIÓN DE DATOS (ETL / FEATURE ENGINEERING)**

In [0]:
# Volver a cargar orders.csv como STRING
df_orders_tbl = spark.read.csv(
    orders_tbl,
    header=True,
    inferSchema=False
)

display(df_orders_tbl)

In [0]:
# 1 FORMATEO DE FECHAS

df_orders_tbl = df_orders_tbl \
    .withColumn(
        "order_date",
        F.to_date(
            F.when(
                F.trim(F.col("order_date")) == "NULL",
                None
            ).otherwise(F.col("order_date"))
        )
    ) \
    .withColumn(
        "required_date",
        F.to_date(
            F.when(
                F.trim(F.col("required_date")) == "NULL",
                None
            ).otherwise(F.col("required_date"))
        )
    ) \
    .withColumn(
        "shipped_date",
        F.to_date(
            F.when(
                F.trim(F.col("shipped_date")) == "NULL",
                None
            ).otherwise(F.col("shipped_date"))
        )
    )

display(df_orders_tbl)

In [0]:
# 2. CREACIÓN DE VARIABLES TEMPORALES

df_orders_tbl = df_orders_tbl \
    .withColumn("anio_orden", F.year("order_date")) \
    .withColumn("mes_orden", F.month("order_date")) \
    .withColumn("dia_orden", F.dayofmonth("order_date")) \
    .withColumn("nombre_mes", F.date_format("order_date", "MMMM")) \
    .withColumn("trimestre", F.quarter("order_date"))

display(df_orders_tbl)

In [0]:
# 3. CÁLCULO DE MONTO TOTAL POR ITEM

df_order_items_tbl = df_order_items_tbl.withColumn(
    "total_item",
    (F.col("quantity") * F.col("list_price")) * (1 - F.col("discount"))
)

display(df_order_items_tbl)

In [0]:
# 4. INTEGRACIÓN DE TABLAS (JOIN TRANSACCIONAL)

df_sales = df_order_items_tbl.alias("oi") \
    .join(df_orders_tbl.alias("o"),
          F.col("oi.order_id") == F.col("o.order_id"),
          "inner") \
    .join(df_products_tbl.alias("p"),
          F.col("oi.product_id") == F.col("p.product_id"),
          "inner") \
    .join(df_categories_tbl.alias("c"),
          F.col("p.category_id") == F.col("c.category_id"),
          "inner") \
    .join(df_brands_tbl.alias("b"),
          F.col("p.brand_id") == F.col("b.brand_id"),
          "inner") \
    .join(df_customers_tbl.alias("cu"),
          F.col("o.customer_id") == F.col("cu.customer_id"),
          "inner") \
    .join(df_stores_tbl.alias("s"),
          F.col("o.store_id") == F.col("s.store_id"),
          "inner")

display(df_sales)

In [0]:
# 5. RENOMBRAR COLUMNAS IMPORTANTES

df_sales = df_sales.select(
    F.col("o.order_id"),
    F.col("o.order_date"),
    F.col("o.order_status"),
    F.col("o.store_id"),

    F.col("cu.customer_id"),
    F.concat_ws(" ", F.col("cu.first_name"), F.col("cu.last_name")).alias("cliente"),

    F.col("p.product_id"),
    F.col("p.product_name"),
    F.col("c.category_name"),
    F.col("b.brand_name"),

    F.col("oi.quantity"),
    F.col("oi.list_price"),
    F.col("oi.discount"),
    F.round(F.col("oi.total_item"), 2).alias("venta_total"),

    F.col("s.store_name"),
    F.col("s.city"),
    F.col("s.state"),

    F.col("o.anio_orden"),
    F.col("o.mes_orden"),
    F.col("o.nombre_mes"),
    F.col("o.trimestre")
)

display(df_sales)

In [0]:
# 6. CREACIÓN DE MÉTRICAS DE NEGOCIO

# Clasificación de descuentos

df_sales = df_sales.withColumn(
    "nivel_descuento",
    F.when(F.col("discount") == 0, "Sin descuento")
     .when(F.col("discount") < 0.10, "Descuento bajo")
     .when(F.col("discount") < 0.20, "Descuento medio")
     .otherwise("Descuento alto")
)

# Clasificación de ventas

df_sales = df_sales.withColumn(
    "categoria_venta",
    F.when(F.col("venta_total") < 500, "Venta pequeña")
     .when(F.col("venta_total") < 2000, "Venta mediana")
     .otherwise("Venta alta")
)

display(df_sales)

In [0]:
# 7. TRANSFORMACIÓN DE STOCKS

df_stocks_tbl = df_stocks_tbl.withColumn(
    "estado_stock",
    F.when(F.col("quantity") == 0, "Sin stock")
     .when(F.col("quantity") < 10, "Stock bajo")
     .otherwise("Stock suficiente")
)

display(df_stocks_tbl)

In [0]:
# 8. TABLA RESUMEN DE VENTAS

df_resumen_ventas = df_sales.groupBy(
    "anio_orden",
    "nombre_mes",
    "category_name"
).agg(
    F.sum("venta_total").alias("ventas_totales"),
    F.sum("quantity").alias("productos_vendidos"),
    F.countDistinct("order_id").alias("cantidad_ordenes")
).orderBy("anio_orden", "ventas_totales", ascending=False)

display(df_resumen_ventas)

In [0]:
# 9. VALIDACIÓN FINAL

print("Cantidad total de registros transformados:")
print(df_sales.count())

print("Estructura final del dataset:")
df_sales.printSchema()

display(df_sales.limit(20))

# **04 REDUCCIÓN DE DIMENSIONALIDAD (PCA)**

In [0]:
# 10. PREPARACIÓN DE FEATURES PARA REDUCCIÓN DE DIMENSIONALIDAD

# Las columnas numéricas disponibles en df_sales que aportan varianza al análisis
# Se excluyen columnas de identificación (IDs) y columnas categóricas/texto

columnas_numericas = [
    "quantity",       # Cantidad de productos por ítem
    "list_price",     # Precio de lista del producto
    "discount",       # Descuento aplicado (0.0 a 1.0)
    "venta_total",    # Monto final de la venta por ítem
    "mes_orden",      # Mes numérico de la orden
    "trimestre"       # Trimestre de la orden
]

# VectorAssembler: fusiona todas las columnas numéricas en un único vector "features_raw"
# El algoritmo PCA (y cualquier algoritmo de MLlib) exige este formato de entrada
assembler_pca = VectorAssembler(
    inputCols=columnas_numericas,
    outputCol="features_raw",
    handleInvalid="skip"   # Descarta filas con nulos residuales en estas columnas
)

# StandardScaler: normaliza cada feature para que todas pesen igual matemáticamente
# withMean=True centra los datos en 0; withStd=True los escala a varianza unitaria
# Sin este paso, "list_price" (valores ~$500-$3000) dominaría sobre "discount" (0.0 a 1.0)
scaler_pca = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)

# Pipeline de preparación (Assembler → Scaler)
pipeline_prep = Pipeline(stages=[assembler_pca, scaler_pca])
df_sales_prep = pipeline_prep.fit(df_sales).transform(df_sales)

print(f"Dataset preparado: {df_sales_prep.count()} registros con vector de {len(columnas_numericas)} dimensiones.")
display(df_sales_prep.select("features_scaled").limit(5))

In [0]:
# 11. REDUCCIÓN DE DIMENSIONALIDAD CON PCA

# PCA (Análisis de Componentes Principales):
# Toma las 6 dimensiones originales y las transforma en un nuevo conjunto de ejes
# llamados "componentes principales", ordenados de mayor a menor varianza explicada.
# El objetivo es quedarnos con los primeros K componentes que concentren
# la mayor parte de la información, eliminando el ruido y la redundancia.

# Fase 1: Entrenamos el modelo PCA con K=6 (igual al total de features)
# para poder observar CUÁNTA varianza explica cada componente antes de reducir
pca_full = PCA(k=6, inputCol="features_scaled", outputCol="pca_features")
modelo_pca_full = pca_full.fit(df_sales_prep)

# Varianza explicada por cada componente individual
varianza_explicada = modelo_pca_full.explainedVariance.toArray()

# Varianza acumulada: nos dice con cuántos componentes capturamos el X% de la info
varianza_acumulada = [float(sum(varianza_explicada[:i+1])) for i in range(len(varianza_explicada))]

print("=======================================================================")
print(" ANÁLISIS DE VARIANZA EXPLICADA POR COMPONENTE PRINCIPAL")
print("=======================================================================")
print(f"{'Componente':<15} {'Varianza Individual':>22} {'Varianza Acumulada':>22}")
print("-" * 60)
for i, (ind, acum) in enumerate(zip(varianza_explicada, varianza_acumulada)):
    print(f"  PC{i+1:<12} {ind*100:>20.2f}%  {acum*100:>20.2f}%")

# Fase 2: Reducción final — nos quedamos con K=3 componentes
# (capturan la mayor parte de la varianza con la mitad de dimensiones)
pca_reducido = PCA(k=3, inputCol="features_scaled", outputCol="pca_features")
modelo_pca_reducido = pca_reducido.fit(df_sales_prep)
df_reducido = modelo_pca_reducido.transform(df_sales_prep)

# Seleccionamos columnas de contexto relevantes junto con los componentes reducidos
df_resultado_pca = df_reducido.select(
    "order_id",
    "product_name",
    "category_name",
    "brand_name",
    "store_name",
    "venta_total",
    "pca_features"
)

varianza_capturada = sum(varianza_explicada[:3]) * 100
print(f"\nReducción aplicada: 6 dimensiones → 3 componentes principales")
print(f"Varianza del dataset capturada con 3 componentes: {varianza_capturada:.2f}%")
print(f"Registros en dataset reducido: {df_resultado_pca.count()}\n")

display(df_resultado_pca.limit(10))

# 05 MODELO DIMENSIONAL (DATA WAREHOUSE)

In [0]:
# ============================================================
# VALIDACIÓN DE INTEGRIDAD REFERENCIAL
# ============================================================

print("===================================================")
print(" VALIDACIÓN DE CALIDAD DE DATOS")
print("===================================================")

# Productos sin categoría

productos_sin_categoria = (
    df_products_tbl.alias("p")
    .join(
        df_categories_tbl.alias("c"),
        F.col("p.category_id") == F.col("c.category_id"),
        "left"
    )
    .filter(F.col("c.category_id").isNull())
)

print("Productos sin categoría:", productos_sin_categoria.count())


# Productos sin marca

productos_sin_marca = (
    df_products_tbl.alias("p")
    .join(
        df_brands_tbl.alias("b"),
        F.col("p.brand_id") == F.col("b.brand_id"),
        "left"
    )
    .filter(F.col("b.brand_id").isNull())
)

print("Productos sin marca:", productos_sin_marca.count())


# Órdenes sin cliente

ordenes_sin_cliente = (
    df_orders_tbl.alias("o")
    .join(
        df_customers_tbl.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .filter(F.col("c.customer_id").isNull())
)

print("Órdenes sin cliente:", ordenes_sin_cliente.count())


# Órdenes sin tienda

ordenes_sin_tienda = (
    df_orders_tbl.alias("o")
    .join(
        df_stores_tbl.alias("s"),
        F.col("o.store_id") == F.col("s.store_id"),
        "left"
    )
    .filter(F.col("s.store_id").isNull())
)

print("Órdenes sin tienda:", ordenes_sin_tienda.count())

In [0]:
# ============================================================
# DIMENSION CLIENTE
# ============================================================

Dim_Cliente = df_customers_tbl.select(
    F.col("customer_id").alias("PK_Cliente"),

    F.concat(
        F.lit("CLI-"),
        F.col("customer_id")
    ).alias("Cod_Cliente"),

    F.concat_ws(
        " ",
        F.col("first_name"),
        F.col("last_name")
    ).alias("Nom_Cliente"),

    F.col("city").alias("Ciudad"),

    F.col("state").alias("Estado")
)

display(Dim_Cliente)

In [0]:
# ============================================================
# DIMENSION PRODUCTO
# ============================================================

Dim_Producto = (
    df_products_tbl.alias("p")

    .join(
        df_categories_tbl.alias("c"),
        F.col("p.category_id") == F.col("c.category_id"),
        "left"
    )

    .join(
        df_brands_tbl.alias("b"),
        F.col("p.brand_id") == F.col("b.brand_id"),
        "left"
    )

    .select(
        F.col("p.product_id").alias("PK_Producto"),

        F.concat(
            F.lit("PROD-"),
            F.col("p.product_id")
        ).alias("Cod_Producto"),

        F.col("p.product_name").alias("Nom_Producto"),

        F.col("c.category_name").alias("Categoria"),

        F.col("b.brand_name").alias("Marca")
    )
)

display(Dim_Producto)

In [0]:
# ============================================================
# DIMENSION TIENDA
# ============================================================

Dim_Tienda = df_stores_tbl.select(
    F.col("store_id").alias("PK_Tienda"),

    F.concat(
        F.lit("STORE-"),
        F.col("store_id")
    ).alias("Cod_Tienda"),

    F.col("store_name").alias("Nom_Tienda"),

    F.col("city").alias("Ciudad"),

    F.col("state").alias("Estado")
)

display(Dim_Tienda)

In [0]:
# ============================================================
# DIMENSION PERSONAL
# ============================================================

Dim_Personal = df_staffs_tbl.select(
    F.col("staff_id").alias("PK_Personal"),

    F.concat(
        F.lit("EMP-"),
        F.col("staff_id")
    ).alias("Cod_Personal"),

    F.concat_ws(
        " ",
        F.col("first_name"),
        F.col("last_name")
    ).alias("Nom_Personal")
)

display(Dim_Personal)

In [0]:
# ============================================================
# DIMENSION TIEMPO
# ============================================================

Dim_Tiempo = (
    df_orders_tbl
    .select("order_date")
    .distinct()

    .withColumn(
        "PK_Tiempo",
        F.date_format(
            "order_date",
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "Fecha",
        F.col("order_date")
    )

    .withColumn(
        "Año",
        F.year("order_date")
    )

    .withColumn(
        "Semestre",
        F.when(
            F.month("order_date") <= 6,
            1
        ).otherwise(2)
    )

    .withColumn(
        "Trimestre",
        F.quarter("order_date")
    )

    .withColumn(
        "Mes",
        F.month("order_date")
    )
)

display(Dim_Tiempo)

In [0]:
# ============================================================
# FACT TABLE VENTAS
# ============================================================

Fact_Ventas = (
    df_order_items_tbl.alias("oi")

    .join(
        df_orders_tbl.alias("o"),
        F.col("oi.order_id") == F.col("o.order_id"),
        "inner"
    )

    .withColumn(
        "Monto_Venta",
        (
            F.col("oi.quantity")
            *
            F.col("oi.list_price")
            *
            (1 - F.col("oi.discount"))
        )
    )

    .select(
        F.col("o.customer_id").alias("FK_Cliente"),

        F.col("oi.product_id").alias("FK_Producto"),

        F.col("o.store_id").alias("FK_Tienda"),

        F.col("o.staff_id").alias("FK_Personal"),

        F.date_format(
            F.col("o.order_date"),
            "yyyyMMdd"
        ).cast("int").alias("FK_Tiempo"),

        F.col("oi.quantity").alias("Cantidad_Vendida"),

        F.round(
            F.col("Monto_Venta"),
            2
        ).alias("Monto_Venta"),

        F.col("oi.discount").alias("Descuento")
    )
)

display(Fact_Ventas)

In [0]:
# ============================================================
# FACT TABLE INVENTARIO
# ============================================================

Fact_Inventario = (
    df_stocks_tbl
    .select(
        F.col("product_id").alias("FK_Producto"),

        F.col("store_id").alias("FK_Tienda"),

        F.col("quantity").alias("Cantidad_Disponible"),

        F.when(
            F.col("quantity") == 0,
            1
        )
        .otherwise(0)
        .alias("Productos_en_Riesgo")
    )
)

display(Fact_Inventario)

In [0]:
# ============================================================
# VALIDACIÓN FINAL
# ============================================================

print("===================================================")
print(" MODELO DIMENSIONAL GENERADO")
print("===================================================")

print("Dim_Cliente      :", Dim_Cliente.count())
print("Dim_Producto     :", Dim_Producto.count())
print("Dim_Tienda       :", Dim_Tienda.count())
print("Dim_Personal     :", Dim_Personal.count())
print("Dim_Tiempo       :", Dim_Tiempo.count())

print("Fact_Ventas      :", Fact_Ventas.count())
print("Fact_Inventario  :", Fact_Inventario.count())